### Qdrant Memory

In [ ]:

class QdrantMemory:
    def __init__(self, qdrant_client, embedding, collection_name=collection_name):
        self.qdrant_client = qdrant_client
        self.embedding = embedding
        self.collection_name = collection_name
        self.qdrant_db = QdrantVectorStore(client=qdrant_client, collection_name=collection_name, embedding=embedding)

    def ensinar(self, pergunta, sql, metadados=None):
        similares = self.qdrant_db.similarity_search(pergunta, k=1)
        if similares:
            print("\n⚠️ Embedding similar já existe. Incrementando score...")
            self.incrementar_score(pergunta)
            return

        vetor = self.embedding.embed_query(pergunta)
        id_unico = str(uuid.uuid4())

        ponto = PointStruct(
            id=id_unico,
            vector=vetor,
            payload={
                "page_content": f"Pergunta: {pergunta}\nSQL: {sql}",
                **(metadados or {}),
                "score": 1
            }
        )
        self.qdrant_client.upsert(collection_name=self.collection_name, points=[ponto])
        print("✅ Embedding armazenado na memoria com sucesso!")

    def incrementar_score(self, pergunta):
        resultado = self.qdrant_client.search(
            collection_name=self.collection_name,
            query_vector=self.embedding.embed_query(pergunta),
            limit=1
        )
        for item in resultado:
            ponto_id = item.id
            payload = item.payload or {}
            print('\nScore: ',self.list_rating(payload["score"], max_score=5))            
            payload["score"] = payload.get("score", 1) + 1
            self.qdrant_client.set_payload(collection_name=self.collection_name, payload=payload, points=[ponto_id])

    def deletar_com_score_menor_que(self, limite_score):
        pontos_para_apagar = self.qdrant_client.scroll(
            collection_name=self.collection_name,
            limit=100,
            filter=Filter(must=[FieldCondition(key="score", match=MatchValue(value=None))])
        )
        for ponto in pontos_para_apagar[0]:
            score = ponto.payload.get("score", 0)
            if score < limite_score:
                self.qdrant_client.delete(collection_name=self.collection_name, points_selector={"points": [ponto.id]})

    def listar_exemplos(self):
        docs = self.qdrant_db.similarity_search("", k=100)
        for i, doc in enumerate(docs):
            print(f"\n🔹 Embedding {i+1}:")
            print(doc.page_content)


    def buscar_com_filtro(self, consulta, filtro):
        docs_filtrados = self.qdrant_db.similarity_search(consulta, k=3, filter=filtro)
        for i, doc in enumerate(docs_filtrados):
            print(f"\n📘 Embedding filtrado {i+1}:")
            print(doc.page_content)
            
    def list_rating(self, score, max_score=5):
        if score:
            min_rating = min(score, max_score)
            score_str = '⭐' * score + '✩' * (max_score - min_rating)
            return score_str
        else:  
            return '✩✩✩✩✩'

    def list_scored_point(self, pergunta):
        
        resultado = self.qdrant_client.search(
            collection_name=self.collection_name,
            query_vector=self.embedding.embed_query(pergunta),
            limit=1
        )
        print("📦 Payload:",resultado)
